# 04 - Entity Resolution
# Enterprise BFSI Client MDM Platform

## Purpose:

   1. Read trusted Silver client records
   2. Generate candidate record pairs using blocking
   3. Apply deterministic matching rules
   4. Apply fuzzy similarity matching
   5. Calculate an explainable composite match score
   6. Classify pairs as MATCH / REVIEW / NO_MATCH
   7. Persist entity-resolution results for stewardship


Blocking:
DOB + postal
surname + DOB
normalized email
phone suffix

Deterministic:
LEI exact
Tax ID exact
Phone + DOB exact
Email + DOB exact

Fuzzy:
name
address
phone
email
DOB

Decision:
MATCH >= 85
REVIEW 70–84
NO MATCH < 70

##Initialize

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "bfsi_mdm_dev"

MATCH_THRESHOLD = 85.0
REVIEW_THRESHOLD = 70.0

print("Entity Resolution notebook initialized.")
print(f"Match threshold  : {MATCH_THRESHOLD}")
print(f"Review threshold : {REVIEW_THRESHOLD}")

Entity Resolution notebook initialized.
Match threshold  : 85.0
Review threshold : 70.0


##Load Silver

In [0]:
silver = spark.table(
    f"{CATALOG}.silver.all_clients"
)

print(
    f"Silver rows: {silver.count():,}"
)

display(
    silver.select(
        "source_system",
        "source_id",
        "normalized_name",
        "normalized_dob",
        "normalized_phone",
        "normalized_email"
    ).limit(10)
)

Silver rows: 2,638,869


source_system,source_id,normalized_name,normalized_dob,normalized_phone,normalized_email
KYC,KYC00000001,Connor Smith,1971-10-19,2074960962,
KYC,KYC00000002,Kevin Ruth Hayes,1970-05-26,1632960381,
KYC,KYC00000003,Jemma Dean,1957-03-03,1174960356,
KYC,KYC00000004,Richard Duncan Chambers,1983-10-21,9098790389,
KYC,KYC00000005,Keith Francis Wilson,1984-05-13,1414960037,
KYC,KYC00000006,Mitchell Janice Blackburn,1974-06-16,1134960059,
KYC,KYC00000007,Raymond Marshall,1988-06-26,2920180580,
KYC,KYC00000008,Clifford Chandler,1974-09-03,1914960258,
KYC,KYC00000009,Dean Reynolds,1978-02-07,1914960387,
KYC,KYC00000010,Aaron Gibson,1980-02-11,1134960631,


##Creating internal record key

In [0]:
silver = silver.withColumn(
    "record_key",
    F.concat_ws(
        "::",
        F.col("source_system"),
        F.col("source_id")
    )
)

##Candidate blocking

Only compare records that have at least one plausible shared characteristic.

Frozen blocking signals:
1. DOB + postal code
2. surname + DOB
3. normalized email
4. phone suffix


###1. Derive postal code

In [0]:
silver = silver.withColumn(
    "postal_code",
    F.upper(
        F.regexp_extract(
            F.col("normalized_address"),
            r"([A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2})$",
            1
        )
    )
)

###2. Derive surname

In [0]:
silver = silver.withColumn(
    "surname",
    F.lower(
        F.element_at(
            F.split(
                F.trim(
                    F.col("normalized_name")
                ),
                r"\s+"
            ),
            -1
        )
    )
)

###3. Derive phone suffix

In [0]:
silver = silver.withColumn(
    "phone_suffix",
    F.when(
        F.length(F.col("normalized_phone")) >= 10,
        F.substring(
            F.col("normalized_phone"),
            -10,
            10
        )
    ).otherwise("")
)

###Build candidate pairs

In [0]:
def build_pairs_on_key(
    df,
    key_column
):
    """
    Generate compact candidate record-key pairs for one blocking key.

    Only record_key and the blocking key are carried through the self-join.
    Blank blocking values are excluded. Same-source pairs are retained.

    IMPORTANT:
    This preserves the project's frozen blocking logic. No arbitrary
    block-size cap is applied because such a cap could introduce false
    negatives and must not be used in final acceptance without validation.
    """

    keyed = (
        df
        .select(
            "record_key",
            key_column
        )
        .filter(
            F.col(key_column).isNotNull()
            & (F.col(key_column) != "")
        )
    )

    left = keyed.select(
        F.col("record_key").alias("record_key_a"),
        F.col(key_column).alias("block_key_a")
    )

    right = keyed.select(
        F.col("record_key").alias("record_key_b"),
        F.col(key_column).alias("block_key_b")
    )

    return (
        left
        .join(
            right,
            (
                F.col("block_key_a")
                ==
                F.col("block_key_b")
            )
            &
            (
                F.col("record_key_a")
                <
                F.col("record_key_b")
            ),
            "inner"
        )
        .select(
            "record_key_a",
            "record_key_b"
        )
    )


###Generate four blocking sets

In [0]:
silver = silver.withColumn(
    "postal_code_dob",
    F.when(
        (
            F.col("postal_code") != ""
        )
        &
        (
            F.col("normalized_dob") != ""
        ),
        F.concat_ws(
            "||",
            "postal_code",
            "normalized_dob"
        )
    ).otherwise("")
)

In [0]:
silver = silver.withColumn(
    "surname_dob",
    F.when(
        (
            F.col("surname") != ""
        )
        &
        (
            F.col("normalized_dob") != ""
        ),
        F.concat_ws(
            "||",
            "surname",
            "normalized_dob"
        )
    ).otherwise("")
)

In [0]:
pairs_postal_dob = build_pairs_on_key(
    silver,
    "postal_code_dob"
)

pairs_surname_dob = build_pairs_on_key(
    silver,
    "surname_dob"
)

pairs_email = build_pairs_on_key(
    silver,
    "normalized_email"
)

pairs_phone = build_pairs_on_key(
    silver,
    "phone_suffix"
)

###Union candidate pairs

In [0]:
candidate_pairs = (
    pairs_postal_dob
    .unionByName(
        pairs_surname_dob
    )
    .unionByName(
        pairs_email
    )
    .unionByName(
        pairs_phone
    )
    .dropDuplicates(
        [
            "record_key_a",
            "record_key_b"
        ]
    )
)

## Materialize compact candidate pairs


In [0]:
# Create MDM schema if it does not exist
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.mdm"
)

# Remove any previous candidate_pairs schema so an older run cannot leave
# stale columns (for example lei_a / tax_id_a) in the Delta table.
spark.sql(
    f"DROP TABLE IF EXISTS {CATALOG}.mdm.candidate_pairs"
)

# Materialize ONLY the compact candidate identifiers.
# Serverless-compatible: no DataFrame cache/persist APIs.
(
    candidate_pairs
    .select(
        "record_key_a",
        "record_key_b"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.mdm.candidate_pairs"
    )
)

candidate_count = spark.table(
    f"{CATALOG}.mdm.candidate_pairs"
).count()

print(
    "Candidate pairs:",
    candidate_count
)


Candidate pairs: 138074372


## Enrich candidate pairs with matching attributes

The blocking table intentionally contains only `record_key_a` and `record_key_b`.
Join those keys back to a compact Silver projection before deterministic/fuzzy matching.
This avoids carrying the full Silver row through the blocking self-joins while providing
the attributes required by downstream matching.

In [0]:
# Build a compact Silver projection containing only fields required for matching.
matching_features = silver.select(
    "record_key",
    "source_system",
    "source_id",
    "normalized_name",
    "normalized_address",
    "normalized_phone",
    "normalized_email",
    "normalized_dob",
    "tax_id",
    "lei"
)

left_features = matching_features.select(
    F.col("record_key").alias("record_key_a"),
    F.col("source_system").alias("source_system_a"),
    F.col("source_id").alias("source_id_a"),
    F.col("normalized_name").alias("normalized_name_a"),
    F.col("normalized_address").alias("normalized_address_a"),
    F.col("normalized_phone").alias("normalized_phone_a"),
    F.col("normalized_email").alias("normalized_email_a"),
    F.col("normalized_dob").alias("normalized_dob_a"),
    F.col("tax_id").alias("tax_id_a"),
    F.col("lei").alias("lei_a")
)

right_features = matching_features.select(
    F.col("record_key").alias("record_key_b"),
    F.col("source_system").alias("source_system_b"),
    F.col("source_id").alias("source_id_b"),
    F.col("normalized_name").alias("normalized_name_b"),
    F.col("normalized_address").alias("normalized_address_b"),
    F.col("normalized_phone").alias("normalized_phone_b"),
    F.col("normalized_email").alias("normalized_email_b"),
    F.col("normalized_dob").alias("normalized_dob_b"),
    F.col("tax_id").alias("tax_id_b"),
    F.col("lei").alias("lei_b")
)

candidate_ids = spark.table(
    f"{CATALOG}.mdm.candidate_pairs"
)

candidate_pairs = (
    candidate_ids
    .join(
        left_features,
        on="record_key_a",
        how="inner"
    )
    .join(
        right_features,
        on="record_key_b",
        how="inner"
    )
)

print("Candidate pairs enriched for matching.")
print("Columns available:", len(candidate_pairs.columns))


Candidate pairs enriched for matching.
Columns available: 20


##Deterministic matching

We apply the strongest identity rules first.
Deterministic rules:
1. LEI exact
2. Tax ID exact
3. Phone + DOB exact
4. Email + DOB exact

In [0]:
candidate_pairs = candidate_pairs.withColumn(
    "deterministic_rule",

    F.when(
        (
            F.col("lei_a") != ""
        )
        &
        (
            F.col("lei_b") != ""
        )
        &
        (
            F.col("lei_a")
            ==
            F.col("lei_b")
        ),
        F.lit("lei_exact")
    )

    .when(
        (
            F.col("tax_id_a") != ""
        )
        &
        (
            F.col("tax_id_b") != ""
        )
        &
        (
            F.col("tax_id_a")
            ==
            F.col("tax_id_b")
        ),
        F.lit("tax_id_exact")
    )

    .when(
        (
            F.col("normalized_phone_a") != ""
        )
        &
        (
            F.col("normalized_dob_a") != ""
        )
        &
        (
            F.col("normalized_phone_a")
            ==
            F.col("normalized_phone_b")
        )
        &
        (
            F.col("normalized_dob_a")
            ==
            F.col("normalized_dob_b")
        ),
        F.lit("phone_dob_exact")
    )

    .when(
        (
            F.col("normalized_email_a") != ""
        )
        &
        (
            F.col("normalized_dob_a") != ""
        )
        &
        (
            F.col("normalized_email_a")
            ==
            F.col("normalized_email_b")
        )
        &
        (
            F.col("normalized_dob_a")
            ==
            F.col("normalized_dob_b")
        ),
        F.lit("email_dob_exact")
    )
)

##Fuzzy similarity

Fuzzy:
name
address
phone
email
DOB

In [0]:
def similarity_score(
    col_a,
    col_b
):
    """
    Convert Levenshtein edit distance into
    an approximate 0-100 similarity score.
    """

    max_length = F.greatest(
        F.length(col_a),
        F.length(col_b)
    )

    return (
        F.when(
            max_length == 0,
            F.lit(100.0)
        )
        .otherwise(
            (
                1.0
                -
                (
                    F.levenshtein(
                        col_a,
                        col_b
                    )
                    /
                    max_length
                )
            )
            * 100.0
        )
    )

###Calculate field-level scores

In [0]:
scored = (

    candidate_pairs

    # --------------------------------------------------------
    # NAME
    # --------------------------------------------------------
    .withColumn(
        "name_score",
        similarity_score(
            F.col(
                "normalized_name_a"
            ),
            F.col(
                "normalized_name_b"
            )
        )
    )

    # --------------------------------------------------------
    # ADDRESS
    # --------------------------------------------------------
    .withColumn(
        "address_score",
        similarity_score(
            F.col(
                "normalized_address_a"
            ),
            F.col(
                "normalized_address_b"
            )
        )
    )

    # --------------------------------------------------------
    # PHONE
    # --------------------------------------------------------
    .withColumn(
        "phone_score",
        F.when(
            (
                F.col("normalized_phone_a")
                == ""
            )
            |
            (
                F.col("normalized_phone_b")
                == ""
            ),
            F.lit(None)
        )
        .when(
            F.col("normalized_phone_a")
            ==
            F.col("normalized_phone_b"),
            F.lit(100.0)
        )
        .otherwise(
            F.lit(0.0)
        )
    )

    # --------------------------------------------------------
    # EMAIL
    # --------------------------------------------------------
    .withColumn(
        "email_score",
        F.when(
            (
                F.col("normalized_email_a")
                == ""
            )
            |
            (
                F.col("normalized_email_b")
                == ""
            ),
            F.lit(None)
        )
        .when(
            F.col("normalized_email_a")
            ==
            F.col("normalized_email_b"),
            F.lit(100.0)
        )
        .otherwise(
            F.lit(0.0)
        )
    )

    # --------------------------------------------------------
    # DOB
    # --------------------------------------------------------
    .withColumn(
        "dob_score",
        F.when(
            (
                F.col("normalized_dob_a")
                == ""
            )
            |
            (
                F.col("normalized_dob_b")
                == ""
            ),
            F.lit(None)
        )
        .when(
            F.col("normalized_dob_a")
            ==
            F.col("normalized_dob_b"),
            F.lit(100.0)
        )
        .otherwise(
            F.lit(0.0)
        )
    )
)

###Weighted explainable score

In [0]:
WEIGHTS = {
    "name_score": 0.35,
    "address_score": 0.20,
    "phone_score": 0.25,
    "email_score": 0.10,
    "dob_score": 0.10,
}

weighted_sum = sum(
    F.coalesce(
        F.col(field) * weight,
        F.lit(0.0)
    )
    for field, weight
    in WEIGHTS.items()
)

weight_total = sum(
    F.when(
        F.col(field).isNotNull(),
        F.lit(weight)
    ).otherwise(
        F.lit(0.0)
    )
    for field, weight
    in WEIGHTS.items()
)

scored = scored.withColumn(
    "fuzzy_score",
    F.when(
        weight_total == 0,
        F.lit(0.0)
    ).otherwise(
        F.round(
            weighted_sum
            /
            weight_total,
            2
        )
    )
)


##Final decision

In [0]:
results = (

    scored

    .withColumn(
        "match_score",
        F.when(
            F.col(
                "deterministic_rule"
            ).isNotNull(),
            F.lit(100.0)
        ).otherwise(
            F.col("fuzzy_score")
        )
    )

    .withColumn(
        "matching_rule",
        F.when(
            F.col(
                "deterministic_rule"
            ).isNotNull(),
            F.col(
                "deterministic_rule"
            )
        ).otherwise(
            F.lit("fuzzy_weighted")
        )
    )

    .withColumn(
        "decision",
        F.when(
            F.col("match_score")
            >= MATCH_THRESHOLD,
            F.lit("MATCH")
        )
        .when(
            F.col("match_score")
            >= REVIEW_THRESHOLD,
            F.lit("REVIEW")
        )
        .otherwise(
            F.lit("NO_MATCH")
        )
    )
)

##Create an explainability reason

In [0]:
results = results.withColumn(
    "match_reason",
    F.when(
        F.col("deterministic_rule").isNotNull(),
        F.concat(
            F.lit("Deterministic rule: "),
            F.col("deterministic_rule")
        )
    ).otherwise(
        F.lit(
            "Fuzzy weighted similarity across "
            "name, address, phone, email and DOB"
        )
    )
)

##Persist match results

In [0]:
(
    results
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.mdm.match_results"
    )
)

##Create the review queue

Only REVIEW records go to human stewardship.

In [0]:
review_queue = (
    results
    .filter(
        F.col("decision")
        ==
        "REVIEW"
    )
    .select(
        "record_key_a",
        "record_key_b",

        "source_system_a",
        "source_id_a",

        "source_system_b",
        "source_id_b",

        "name_score",
        "address_score",
        "phone_score",
        "email_score",
        "dob_score",

        "match_score",
        "matching_rule",
        "match_reason",

        F.current_timestamp().alias(
            "review_created_at"
        )
    )
)

In [0]:
(
    review_queue
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.mdm.review_queue"
    )
)

##Entity Resolution verification

In [0]:
spark.sql(
    f"""
    SELECT
        decision,
        COUNT(*) AS count
    FROM {CATALOG}.mdm.match_results
    GROUP BY decision
    ORDER BY decision
    """
).show()

+--------+---------+
|decision|    count|
+--------+---------+
|   MATCH|  2249077|
|NO_MATCH|135665735|
|  REVIEW|   159560|
+--------+---------+



In [0]:
spark.sql(
    f"""
    SELECT
        matching_rule,
        COUNT(*) AS count
    FROM {CATALOG}.mdm.match_results
    GROUP BY matching_rule
    ORDER BY count DESC
    """
).show()

+---------------+---------+
|  matching_rule|    count|
+---------------+---------+
| fuzzy_weighted|136026590|
|phone_dob_exact|  1961486|
|email_dob_exact|    70298|
|   tax_id_exact|    14567|
|      lei_exact|     1431|
+---------------+---------+



In [0]:
spark.sql(
    f"""
    SELECT
        MIN(match_score),
        MAX(match_score),
        AVG(match_score)
    FROM {CATALOG}.mdm.match_results
    """
).show()

+----------------+----------------+-----------------+
|MIN(match_score)|MAX(match_score)| AVG(match_score)|
+----------------+----------------+-----------------+
|            10.0|           100.0|38.32296971844999|
+----------------+----------------+-----------------+

